In [1]:
# 1. IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler
)

from sklearn.svm import SVR

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.neural_network import ( MLPRegressor )

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

In [2]:
# 2. LOAD DATASET

df = pd.read_csv("houses_improved.csv")

In [3]:
# 3. CHECK DATASET

print("First Five Rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

First Five Rows:
   Number_of_Rooms  Site_Area_sqm  Built_Area_sqm  Property_Years  \
0                3            402             275              18   
1                6            111              72               5   
2                4            417             306               2   
3                4            179              72               4   
4                3            315             147              12   

  Construction_Materials Housing_Typology Land_Value_Grading  \
0               Concrete    Semi-detached                Low   
1               Concrete    Semi-detached               High   
2               Concrete      Condominium             Medium   
3               Mud&Wood      Condominium                Low   
4               Concrete    Semi-detached             Medium   

   Proximity_to_CBD_km  Proximity_to_Bus_Station_km Type_of_Nearest_Road  \
0                 0.66                         0.39               Gravel   
1                 0.12         

In [4]:
# 4. SEPARATE FEATURES AND TARGET

y = df["Price_ETB"]

X = df.drop(
    "Price_ETB",
    axis=1
)

In [5]:
# 5. DEFINE CATEGORICAL COLUMNS

categorical_columns = [

    "Construction_Materials",

    "Housing_Typology",

    "Land_Value_Grading",

    "Type_of_Nearest_Road"

]

In [6]:
# 6. LABEL ENCODING
X_label = X.copy()

label_encoders = {}

for col in categorical_columns:

    encoder = LabelEncoder()

    X_label[col] = encoder.fit_transform(
        X_label[col]
    )

    label_encoders[col] = encoder


print("\nLabel Encoded Data:")
print(X_label.head())


Label Encoded Data:
   Number_of_Rooms  Site_Area_sqm  Built_Area_sqm  Property_Years  \
0                3            402             275              18   
1                6            111              72               5   
2                4            417             306               2   
3                4            179              72               4   
4                3            315             147              12   

   Construction_Materials  Housing_Typology  Land_Value_Grading  \
0                       0                 2                   1   
1                       0                 2                   0   
2                       0                 0                   2   
3                       1                 0                   1   
4                       0                 2                   2   

   Proximity_to_CBD_km  Proximity_to_Bus_Station_km  Type_of_Nearest_Road  \
0                 0.66                         0.39                     1   
1      

In [7]:
# 7. ONE-HOT ENCODING

X_onehot = pd.get_dummies(

    X,

    columns=categorical_columns,

    drop_first=True

)


print("\nOne-Hot Encoded Data:")
print(X_onehot.head())


One-Hot Encoded Data:
   Number_of_Rooms  Site_Area_sqm  Built_Area_sqm  Property_Years  \
0                3            402             275              18   
1                6            111              72               5   
2                4            417             306               2   
3                4            179              72               4   
4                3            315             147              12   

   Proximity_to_CBD_km  Proximity_to_Bus_Station_km  Proximity_to_Schools_km  \
0                 0.66                         0.39                     1.51   
1                 0.12                         1.38                     2.04   
2                 5.00                         2.21                     2.71   
3                 4.03                         1.08                     1.30   
4                 3.95                         1.26                     2.89   

   Construction_Materials_Mud&Wood  Housing_Typology_Detached  \
0               

In [8]:
# 8. TRAIN-TEST SPLIT

X_label_train, X_label_test, y_train, y_test = train_test_split(

    X_label,

    y,

    test_size=0.2,

    random_state=42

)


X_onehot_train, X_onehot_test, _, _ = train_test_split(

    X_onehot,

    y,

    test_size=0.2,

    random_state=42

)

In [9]:
# 9. DEFINE MODELS

models = {

    "SVR": SVR(),

    "Random Forest":
        RandomForestRegressor(
            random_state=42
        ),
    
      "MLP Regressor":
      MLPRegressor(
        random_state=42,
         max_iter=3000,
           early_stopping=True
        ),
    
    "Gradient Boosting":
        GradientBoostingRegressor(
            random_state=42
        ),

    "Linear Regression":
        LinearRegression(),

    "Ridge":
        Ridge(),

    "Lasso":
        Lasso(
            random_state=42
        )

}

In [10]:
# 10. DEFINE SCALING METHODS

scalers = {

    "No Scaling": None,

    "StandardScaler":
        StandardScaler(),

    "MinMaxScaler":
        MinMaxScaler()

}

In [11]:
# 11. DEFINE DATASETS
datasets = {

    "Label Encoding": (

        X_label_train,

        X_label_test

    ),

    "One-Hot Encoding": (

        X_onehot_train,

        X_onehot_test

    )

}

In [12]:
# 12. RUN EXPERIMENTS

results = []

for encoding_name, (X_train, X_test) in datasets.items():

    for scaling_name, scaler in scalers.items():

        X_train_processed = X_train.copy()
        X_test_processed = X_test.copy()

        if scaler is not None:
            scaler.fit(X_train_processed)

            X_train_processed = scaler.transform(X_train_processed)
            X_test_processed = scaler.transform(X_test_processed)

        for model_name, model in models.items():

            model.fit(X_train_processed, y_train)

            predictions = model.predict(X_test_processed)

            r2 = r2_score(y_test, predictions)

            rmse = np.sqrt(mean_squared_error(y_test, predictions))

            mae = mean_absolute_error(y_test, predictions)

            results.append({
                "Encoding": encoding_name,
                "Scaling": scaling_name,
                "Model": model_name,
                "R2 Score": r2,
                "RMSE": rmse,
                "MAE": mae
            })

In [13]:
# 13. CREATE RESULTS DATAFRAME

results_df = pd.DataFrame(

    results

)


In [14]:
# 14. SORT RESULTS

results_df = results_df.sort_values(

    by="R2 Score",

    ascending=False

)

In [15]:
# 15. DISPLAY ALL RESULTS

print("\nALL MODEL RESULTS:")

print(

    results_df.to_string(

        index=False

    )

)


ALL MODEL RESULTS:
        Encoding        Scaling             Model  R2 Score         RMSE          MAE
One-Hot Encoding StandardScaler Gradient Boosting  0.898719 4.095998e+05 3.126913e+05
One-Hot Encoding   MinMaxScaler Gradient Boosting  0.898704 4.096297e+05 3.128088e+05
One-Hot Encoding     No Scaling Gradient Boosting  0.898181 4.106845e+05 3.126665e+05
  Label Encoding StandardScaler Gradient Boosting  0.896844 4.133733e+05 3.190019e+05
  Label Encoding   MinMaxScaler Gradient Boosting  0.896707 4.136463e+05 3.191678e+05
  Label Encoding     No Scaling Gradient Boosting  0.896124 4.148121e+05 3.194070e+05
One-Hot Encoding StandardScaler     Random Forest  0.846433 5.043634e+05 3.914624e+05
One-Hot Encoding   MinMaxScaler     Random Forest  0.846148 5.048315e+05 3.916727e+05
One-Hot Encoding     No Scaling     Random Forest  0.846141 5.048428e+05 3.920723e+05
  Label Encoding   MinMaxScaler     Random Forest  0.836120 5.210243e+05 4.082545e+05
  Label Encoding     No Scaling   

In [16]:
# 16. DISPLAY BEST TECHNIQUE

best_result = results_df.iloc[0]


print("\nBEST TECHNIQUE:")

print(best_result)


BEST TECHNIQUE:
Encoding     One-Hot Encoding
Scaling        StandardScaler
Model       Gradient Boosting
R2 Score             0.898719
RMSE            409599.761562
MAE              312691.32284
Name: 31, dtype: object


In [17]:
import joblib

best_model = GradientBoostingRegressor(random_state=42)

best_model.fit(X_onehot_train, y_train)

joblib.dump(best_model, "model.pkl")

['model.pkl']

In [18]:
joblib.dump(X_onehot.columns.tolist(), "features.pkl")

['features.pkl']

In [19]:
from flask import Flask, render_template, request
import joblib
import pandas as pd

app = Flask(__name__)

model = joblib.load("model.pkl")
features = joblib.load("features.pkl")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():

    data = {
        "Number_of_Rooms": float(request.form["rooms"]),
        "Site_Area_sqm": float(request.form["site"]),
        "Built_Area_sqm": float(request.form["built"]),
        "Property_Years": float(request.form["years"]),
        "Construction_Materials": request.form["material"],
        "Housing_Typology": request.form["type"],
        "Land_Value_Grading": request.form["grading"],
        "Proximity_to_CBD_km": float(request.form["cbd"]),
        "Proximity_to_Bus_Station_km": float(request.form["bus"]),
        "Type_of_Nearest_Road": request.form["road"],
        "Proximity_to_Schools_km": float(request.form["school"])
    }

    df = pd.DataFrame([data])

    df = pd.get_dummies(df)

    df = df.reindex(columns=features, fill_value=0)

    prediction = model.predict(df)[0]

    return render_template(
        "index.html",
        prediction=round(prediction, 2)
    )

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\ISUG\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [37]:
!pip install flask

In [ ]:
!python app.py